2.3.1 Импорты, seed и device

In [4]:
!pip install torch
!pip install torchvision
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, random_split

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\korobochka_sahara\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
cpu



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\korobochka_sahara\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


2.3.2. Данные и DataLoader

In [5]:
transform = transforms.ToTensor()

dataset = torchvision.datasets.EMNIST(
    root="./data",
    split='balanced',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.EMNIST(
    root="./data",
    split='balanced',
    train=False,
    download=True,
    transform=transform
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

x, y = next(iter(train_loader))

print(x.shape)
print(y.shape)
print(x.min(), x.max())

torch.Size([128, 1, 28, 28])
torch.Size([128])
tensor(0.) tensor(1.)


2.3.3. Модель MLP и цикл обучения

In [6]:
class MLP(nn.Module):

    def __init__(self, hidden_sizes=[256,128], num_classes=10, dropout=0.0, batchnorm=False):
        super().__init__()

        layers = []
        input_size = 28*28

        for h in hidden_sizes:
            layers.append(nn.Linear(input_size, h))

            if batchnorm:
                layers.append(nn.BatchNorm1d(h))

            layers.append(nn.ReLU())

            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            input_size = h

        layers.append(nn.Linear(input_size, num_classes))

        self.net = nn.Sequential(
            nn.Flatten(),
            *layers
        )

    def forward(self, x):
        return self.net(x)  

Функции обучения

In [7]:
def train_one_epoch(model, loader, optimizer, criterion):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for x, y in loader:

        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * x.size(0)

        preds = logits.argmax(dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss/total, correct/total

def evaluate(model, loader, criterion):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in loader:

            x, y = x.to(device), y.to(device)

            logits = model(x)

            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)

            preds = logits.argmax(1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss/total, correct/total

Цикл обучения

In [8]:
def train_model(model, train_loader, val_loader, optimizer, criterion, epochs):

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    for epoch in range(epochs):

        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)

        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(epoch, train_loss, val_loss, train_acc, val_acc)

    return history

3.1. Часть A (S08): регуляризация и переобучение

E1 base

In [9]:
model = MLP(
    hidden_sizes=[512,256,128],
    num_classes=47,
    dropout=0,
    batchnorm=False
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)
history_E1 = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs=15
)

0 1.1610534577082234 0.6920475990214247 0.6575243794326241 0.7825797872340425
1 0.6070410422822262 0.5757229298564559 0.8000443262411348 0.8111702127659575
2 0.49735401209364544 0.5452961526863964 0.8300864361702127 0.8207003546099291
3 0.4352697695823426 0.4978681646340282 0.8471187943262412 0.8341312056737589
4 0.3914703043851447 0.47654559523501294 0.8614694148936171 0.8427304964539007
5 0.35623098375103995 0.4761969423463159 0.8696919326241135 0.8451684397163121
6 0.3285069626273838 0.4900245406103472 0.8777814716312057 0.8318705673758865
7 0.30207248554162097 0.490197754328978 0.8858931737588652 0.8444148936170213
8 0.2804627519972781 0.47806826348000386 0.8926750886524822 0.8496897163120567
9 0.2606359621100392 0.4937271504537434 0.8978723404255319 0.848404255319149
10 0.24657818481643148 0.5189821825805285 0.902415780141844 0.8434397163120567
11 0.23303390204483734 0.5274063876334657 0.9060948581560284 0.8456117021276596
12 0.21702405864253957 0.5415769935076964 0.91274379432624

E2 Dropout

In [10]:
model = MLP(
    hidden_sizes=[512,256,128],
    num_classes=47,
    dropout=0.3,
    batchnorm=False
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history_E2 = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs=15
)

0 1.5023564468038844 0.7471968363362846 0.560372340425532 0.7640070921985815
1 0.847897943960014 0.6253126475827914 0.7329233156028369 0.799290780141844
2 0.7240638413327806 0.5406902331832453 0.7675864361702127 0.8173315602836879
3 0.6536975420113151 0.5250863983276043 0.7854831560283688 0.8245124113475177
4 0.612093197453952 0.49744148761668106 0.7960992907801419 0.8303191489361702
5 0.584493372922248 0.48246756885068637 0.8044104609929078 0.8312056737588652
6 0.5640886926904638 0.47490489347606685 0.8099179964539007 0.8403812056737588
7 0.538476104643328 0.46058240308829235 0.8167220744680851 0.8426861702127659
8 0.5239402645023157 0.4570820726824145 0.8213098404255319 0.8433067375886525
9 0.5094514528064863 0.4474678715070089 0.8254100177304965 0.850177304964539
10 0.4996465249264494 0.4464956259051113 0.8275487588652483 0.8484929078014184
11 0.4872591586823159 0.44688333877434966 0.8307734929078014 0.8492021276595745
12 0.47647824762983526 0.4383256512330779 0.8350842198581561 0.8

E3 BatchNorm

In [11]:
model = MLP(
    hidden_sizes=[512,256,128],
    num_classes=47,
    dropout=0,
    batchnorm=True
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history_E3 = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs=15
)

0 0.8918576084552927 0.5568482912601309 0.7480939716312057 0.816622340425532
1 0.485877547746009 0.46996808500154647 0.8347960992907801 0.8410017730496454
2 0.40882810757938004 0.45900560697765214 0.8547539893617021 0.844459219858156
3 0.36338584615406416 0.4418175591644666 0.8675310283687944 0.8502659574468086
4 0.32701220918208995 0.42669068207132055 0.8784020390070922 0.8530585106382979
5 0.2996166304704991 0.4343395932769099 0.8860815602836879 0.8529698581560283
6 0.28151335384406095 0.4270956614761488 0.8916999113475177 0.8557624113475177
7 0.260191651153649 0.43685632948334335 0.8981050531914894 0.8583333333333333
8 0.24385356991849047 0.46103459186587775 0.9032136524822695 0.8481382978723404
9 0.22919382839760882 0.4393172635677013 0.907457890070922 0.8580230496453901
10 0.21598676617475265 0.4649632264536323 0.9129654255319148 0.8506648936170212
11 0.2043096864054389 0.4651222104721881 0.9170434397163121 0.8538120567375886
12 0.1958438328500335 0.47943859713297365 0.91945921985

E4 EarlyStopping

In [12]:
model = MLP(
    hidden_sizes=[512,256,128],
    num_classes=47,
    dropout=0.3,
    batchnorm=False
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_val_loss = float("inf")
patience = 4
counter = 0

history_E4 = {
    "train_loss": [],
    "val_loss": []
}

for epoch in range(20):

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    history_E4["train_loss"].append(train_loss)
    history_E4["val_loss"].append(val_loss)

    print(epoch, train_loss, val_loss, val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0

        torch.save(model.state_dict(), "artifacts/best_model.pt")

    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered")
        break

0 1.4929665187571912 0.7383585094560122 0.7670212765957447
1 0.8399909949471764 0.5975270600183635 0.8081560283687943
2 0.7160897351325827 0.5437518904395138 0.819636524822695
3 0.6530377543987111 0.5140444951699981 0.8239804964539007
4 0.6132485317422989 0.48452288977643276 0.8374113475177305
5 0.5810042486123159 0.48646411684387963 0.8342198581560284
6 0.556189319047522 0.46405648187542636 0.8425531914893617
7 0.5398953272095809 0.47611163524871175 0.8406471631205674
8 0.5209425233357342 0.4458491768820066 0.8480496453900709
9 0.5093726740661242 0.4398107166831375 0.8502216312056737
10 0.49612179529582356 0.44450854688671465 0.8479166666666667
11 0.48806162763994637 0.4441140323665971 0.8485372340425532
12 0.47844844959306376 0.4344511437077894 0.8511524822695036
13 0.4696595667101813 0.43538340440033174 0.8495124113475178
14 0.4649871386323415 0.43545791109402976 0.8498670212765957
15 0.4532794212618618 0.43624945718345914 0.8519503546099291
16 0.4432849872830912 0.4275520372052565 

3.2. Часть B (S09): LR, оптимизаторы, weight decay

**O1 (LR слишком большой)**

In [13]:
model = MLP(
    hidden_sizes=[512,256,128],
    num_classes=47,
    dropout=0.3,
    batchnorm=False
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-1
)
history_O1 = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs=6
)

0 7.0940800156153685 3.85918055933418 0.02078900709219858 0.022650709219858157
1 3.8644914532384127 3.8592176133013787 0.02074468085106383 0.02149822695035461
2 3.8865926772990127 3.8624745984449453 0.020966312056737588 0.021409574468085106
3 3.8647467427219904 3.859143647741764 0.02198581560283688 0.021453900709219858
4 4.364876051152006 3.8614881620339467 0.02056737588652482 0.021187943262411347
5 3.8715172656038974 3.867895938318672 0.02106604609929078 0.020434397163120566


**O2 (LR слишком маленький)**

In [14]:
model = MLP(
    hidden_sizes=[512,256,128],
    num_classes=47,
    dropout=0.3,
    batchnorm=False
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-5
)
history_O2 = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs=6
)

0 3.8255329358662276 3.753787870948196 0.05140735815602837 0.16103723404255318
1 3.595278348314001 3.300876584966132 0.11432845744680852 0.23209219858156027
2 3.183408160243474 2.8285155685235424 0.16411790780141844 0.3350177304964539
3 2.8742423429556774 2.511643761438681 0.22217420212765956 0.39986702127659574
4 2.662690569153914 2.2961523130430397 0.26437278368794326 0.43816489361702127
5 2.50766897472084 2.1381591218583127 0.2992796985815603 0.4659131205673759


**O3 (SGD+momentum + weight decay)**

In [15]:
model = MLP(
    hidden_sizes=[512,256,128],
    num_classes=47,
    dropout=0.3,
    batchnorm=False
).to(device)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=1e-4
)
history_O3 = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs=12
)

0 2.961038794754245 1.5156481889968223 0.2144503546099291 0.5707890070921986
1 1.412849328703914 0.9533380981878186 0.5792774822695036 0.7101063829787234
2 1.0656835687921402 0.7642396928570795 0.6718860815602837 0.763031914893617
3 0.9093949798996567 0.6810524349517011 0.7128656914893617 0.781072695035461
4 0.8201690579982515 0.6259539494277737 0.7401706560283688 0.7993351063829788
5 0.7540133540089249 0.5750055591265361 0.7574246453900709 0.8112145390070922
6 0.709245923816735 0.5487522700999645 0.7708000886524823 0.8188829787234042
7 0.6758120236244608 0.5288601981832626 0.7816932624113475 0.8238475177304965
8 0.64634866177613 0.5130730837794906 0.787854609929078 0.8296985815602836
9 0.6188232422720455 0.49961507396495086 0.795700354609929 0.830540780141844
10 0.6006107002708083 0.4994295214930325 0.7998670212765957 0.8281028368794326
11 0.5802352627963885 0.4761553923711709 0.805718085106383 0.8404698581560284


4. Артефакты

In [16]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

# curves_best.png

plt.figure()

plt.plot(history_E4["train_loss"], label="train loss")
plt.plot(history_E4["val_loss"], label="val loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Best model training curves")
plt.legend()

plt.savefig("artifacts/figures/curves_best.png")
plt.close()


# curves_lr_extremes.png

plt.figure()

plt.plot(history_O1["train_loss"], label="O1 large LR")
plt.plot(history_O2["train_loss"], label="O2 small LR")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning rate extremes")
plt.legend()

plt.savefig("artifacts/figures/curves_lr_extremes.png")
plt.close()


# best_config.json

best_config = {
    "dataset": "EMNIST",
    "seed": 42,
    "hidden_sizes": [512,256,128],
    "activation": "ReLU",
    "dropout": 0.3,
    "batchnorm": False,
    "optimizer": "Adam",
    "lr": 1e-3
}

with open("artifacts/best_config.json", "w") as f:
    json.dump(best_config, f, indent=4)


# runs.csv

best_O1_acc = max(history_O1["val_acc"])
best_O1_loss = min(history_O1["val_loss"])

best_O2_acc = max(history_O2["val_acc"])
best_O2_loss = min(history_O2["val_loss"])

best_O3_acc = max(history_O3["val_acc"])
best_O3_loss = min(history_O3["val_loss"])

runs = [
["E1","EMNIST",42,"512-256-128 ReLU","Adam",1e-3,0,0,15,0.851,0.469],
["E2","EMNIST",42,"512-256-128 + Dropout(0.3)","Adam",1e-3,0,0,11,0.851,0.436],
["E3","EMNIST",42,"512-256-128 + BatchNorm","Adam",1e-3,0,0,15,0.856,0.433],
["E4","EMNIST",42,"Dropout + EarlyStopping","Adam",1e-3,0,0,len(history_E4["train_loss"]),0.857,best_val_loss],
["O1","EMNIST",42,"same as E4","Adam",1e-1,0,0,6,best_O1_acc,best_O1_loss],
["O2","EMNIST",42,"same as E4","Adam",1e-5,0,0,6,best_O2_acc,best_O2_loss],
["O3","EMNIST",42,"same as E4","SGD",0.01,0.9,1e-4,12,best_O3_acc,best_O3_loss]
]

columns = [
"experiment_id","dataset","seed","model_summary",
"optimizer","lr","momentum","weight_decay",
"epochs_trained","best_val_accuracy","best_val_loss"
]

df = pd.DataFrame(runs, columns=columns)

df.to_csv("artifacts/runs.csv", index=False)